In [1]:
%pylab inline
import gc
import fitsio
import numpy as np
import astropy.io.fits as pyfits
import astropy.io.ascii as ascii
import astropy.table as astTable
import anautil

Populating the interactive namespace from numpy and matplotlib


In [2]:
cd /work/xiangchong.li/work/cosmicShear/s16a

/work/xiangchong.li/work/cosmicShear/s16a


In [3]:
fname='/work/xiangchong.li/cosmosis/cosmosis-standard-library/likelihood/cfhtlens/cfhtlens_heymans13.fits'
hdus=pyfits.open(fname)
hdu0=pyfits.PrimaryHDU()

In [4]:
# only use the xi_+
dd=hdus[1].data[:105,:105]
# only use redshift bins 1~4
dd2=hdus[2].data
msk=(dd2['BIN1']<5)&(dd2['BIN2']<5)
# same for covariance
dd=dd[msk,:]
dd=dd[:,msk]
### 
### covariance
###
# initiate the header
hd1=pyfits.Header()
hd1['EXTNAME']='COVMAT'
hd1['NAME_0']='xi_plus'
hd1['STRT_0']=0
hdu1=pyfits.hdu.ImageHDU(data=dd,header=hd1)
hdu12=anautil.make_cscov_hdu(dd)
assert hdu12.header==hdu1.header
assert np.all(hdu12.data==hdu1.data)

###
### data
###
dd=fitsio.read(fname,ext=2)
msk=(dd['BIN1']<5)&(dd['BIN2']<5)
dataC=dd[msk]
hd2=pyfits.Header()
hd2['EXTNAME']='xi_plus'
hd2['2PTDATA ']='T'
hd2['WINDOWS']='SAMPLE'
hd2['QUANT1']='G+R'
hd2['QUANT2']='G+R'
hd2['KERNEL_1']='NZ_SAMPLE'
hd2['KERNEL_2']='NZ_SAMPLE'

hdu2=pyfits.hdu.BinTableHDU(data=dataC,header=hd2)
hdu2.header['TUNIT5']='arcmin'
hdu22=anautil.make_cs_hdu(dataC)
assert np.all(hdu22.data==hdu2.data)
#hdul = pyfits.HDUList([hdu0,hdu1,hdu2])
#hdul.writeto('data/cfhtlens_heymans13.fits',overwrite=True)

In [5]:
dd=fitsio.read(fname,ext=4)
names=['Z_LOW','Z_MID','Z_HIGH','BIN1','BIN2','BIN3','BIN4']
names2=['BIN1','BIN2','BIN3','BIN4']
nzList=[]
for i in range(4):
    nzList.append(dd[names2[i]])
hdu32=anautil.make_nz_hdu(nzList,dd['Z_MID'])
np.testing.assert_allclose(hdu32.data['Z_LOW'],dd['Z_LOW'],rtol=1e-5,atol=1e-5)
np.testing.assert_allclose(hdu32.data['Z_HIGH'],dd['Z_HIGH'],rtol=1e-5,atol=1e-5)
np.testing.assert_allclose(hdu32.data['Z_MID'],dd['Z_MID'],rtol=1e-5,atol=1e-5)
np.testing.assert_allclose(hdu32.data['BIN1'],dd['BIN1'],rtol=1e-5,atol=1e-5)
#pyfits.writeto('photoz/cfhtlens_heymans13.fits',data=dataC,header=hdus[4].header,overwrite=True)

0.050000002999999994
